In [0]:
files = dbutils.fs.ls("abfss://tokyo-olympic-data@tokyoolympichaseebdata.dfs.core.windows.net/raw-data/")

dfs = {}

for f in files:
    if f.path.endswith(".csv"):
        df_name = f.name.replace(".csv", "")
        
        dfs[df_name] = spark.read \
            .format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(f.path)



In [0]:
display(dfs['medals'])

In [0]:
#Top countries that won Gold medals
top_gold_count = dfs['medals'].orderBy('Gold',ascending=False).select('TeamCountry','Gold').show()

In [0]:
#Avg entries Male and Female for each discipline

avg_entries_gender = dfs['entriesgender'].withColumn(
    'Avg_Male', dfs['entriesgender']['Male'] / dfs['entriesgender']['Total']
).withColumn(
    'Avg_Female', dfs['entriesgender']['Female'] / dfs['entriesgender']['Total']
).show()

In [0]:
base_path = "abfss://tokyo-olympic-data@tokyoolympichaseebdata.dfs.core.windows.net/transformed-data/"

for name, df in dfs.items():
    
    output_path = f"{base_path}{name}/"
    
    df.write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(output_path)
    
    print(f"Written: {name} → {output_path}")